Imports and configuration

In [22]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

SEED = 42

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

DATASET_DIR = Path("../data/raw/garbage-dataset")
MANIFEST_DIR = Path("../data/manifests")

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_DIR.resolve())
print("Manifest directory:", MANIFEST_DIR.resolve())

print("\nSplit configuration:")
print(f"Train: {TRAIN_RATIO:.0%}")
print(f"Validation: {VAL_RATIO:.0%}")
print(f"Test: {TEST_RATIO:.0%}")
print(f"Random seed: {SEED}")

Dataset: C:\Users\USER\Desktop\SLIIT\DL\Assignment\waste-classification\data\raw\garbage-dataset
Manifest directory: C:\Users\USER\Desktop\SLIIT\DL\Assignment\waste-classification\data\manifests

Split configuration:
Train: 70%
Validation: 15%
Test: 15%
Random seed: 42


Verify ratios

In [23]:
assert abs(
    TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0
) < 1e-8, "Split ratios must sum to 1."

assert DATASET_DIR.exists(), (
    f"Dataset directory not found: {DATASET_DIR.resolve()}"
)

print("Configuration validation passed.")

Configuration validation passed.


Rebuild the image inventory

In [24]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

records = []

class_dirs = sorted([
    folder
    for folder in DATASET_DIR.iterdir()
    if folder.is_dir()
])

for class_dir in class_dirs:

    class_name = class_dir.name

    for file_path in sorted(class_dir.iterdir()):

        if (
            file_path.is_file()
            and file_path.suffix.lower() in IMAGE_EXTENSIONS
        ):

            relative_path = file_path.relative_to(DATASET_DIR)

            records.append({
                "relative_path": relative_path.as_posix(),
                "class_name": class_name
            })

df = pd.DataFrame(records)

print("Total images:", len(df))
print("Number of classes:", df["class_name"].nunique())

display(df.head())

Total images: 19762
Number of classes: 10


,relative_path,class_name
0,battery/battery_1.jpg,battery
1,battery/battery_10.jpg,battery
2,battery/battery_100.jpg,battery
3,battery/battery_101.jpg,battery
4,battery/battery_102.jpg,battery


Verify dataset before splitting

In [25]:
expected_classes = {
    "battery",
    "biological",
    "cardboard",
    "clothes",
    "glass",
    "metal",
    "paper",
    "plastic",
    "shoes",
    "trash"
}

actual_classes = set(df["class_name"].unique())

assert len(df) == 19762, (
    f"Expected 19,762 images but found {len(df)}"
)

assert actual_classes == expected_classes, (
    f"Unexpected classes: {actual_classes}"
)

assert df["relative_path"].is_unique, (
    "Duplicate relative paths detected."
)

print("Dataset validation passed.")

Dataset validation passed.


Train vs temporary

In [26]:
train_df, temp_df = train_test_split(
    df,
    test_size=(VAL_RATIO + TEST_RATIO),
    random_state=SEED,
    stratify=df["class_name"],
    shuffle=True
)

print("Training images:", len(train_df))
print("Temporary images:", len(temp_df))

Training images: 13833
Temporary images: 5929


Split temporary data into validation/test

In [27]:
relative_test_ratio = TEST_RATIO / (
    VAL_RATIO + TEST_RATIO
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_ratio,
    random_state=SEED,
    stratify=temp_df["class_name"],
    shuffle=True
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print(
    "\nTotal:",
    len(train_df) + len(val_df) + len(test_df)
)

Train: 13833
Validation: 2964
Test: 2965

Total: 19762


Add split labels

In [28]:
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

print(train_df.head())

                  relative_path class_name  split
6004   clothes/clothes_3010.jpg    clothes  train
11816       glass/glass_561.jpg      glass  train
5094   clothes/clothes_2191.jpg    clothes  train
16607   plastic/plastic_732.jpg    plastic  train
10109      glass/glass_2512.jpg      glass  train


Verify zero overlap

In [29]:
train_paths = set(train_df["relative_path"])
val_paths = set(val_df["relative_path"])
test_paths = set(test_df["relative_path"])

train_val_overlap = train_paths.intersection(val_paths)
train_test_overlap = train_paths.intersection(test_paths)
val_test_overlap = val_paths.intersection(test_paths)

print(
    "Train ↔ Validation overlap:",
    len(train_val_overlap)
)

print(
    "Train ↔ Test overlap:",
    len(train_test_overlap)
)

print(
    "Validation ↔ Test overlap:",
    len(val_test_overlap)
)

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

print("\nNo split overlap detected.")

Train ↔ Validation overlap: 0
Train ↔ Test overlap: 0
Validation ↔ Test overlap: 0

No split overlap detected.


Verify no images disappeared

In [30]:
all_split_paths = (
    train_paths
    | val_paths
    | test_paths
)

original_paths = set(df["relative_path"])

assert all_split_paths == original_paths

assert (
    len(train_df)
    + len(val_df)
    + len(test_df)
    == len(df)
)

print("All images accounted for.")
print("Total:", len(all_split_paths))

All images accounted for.
Total: 19762


Inspect class distribution in every split

In [31]:
combined_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

split_counts = pd.crosstab(
    combined_df["class_name"],
    combined_df["split"]
)

split_counts = split_counts[
    ["train", "val", "test"]
]

split_counts["total"] = split_counts.sum(axis=1)

display(split_counts)

split,train,val,test,total
class_name,,,,
battery,661,142,141,944
biological,698,150,149,997
cardboard,1277,274,274,1825
clothes,3729,799,799,5327
glass,2142,459,460,3061
metal,714,153,153,1020
paper,1176,252,252,1680
plastic,1389,297,298,1984
shoes,1384,296,297,1977


Calculate percentages

In [32]:
split_percentages = (
    split_counts[["train", "val", "test"]]
    .div(split_counts["total"], axis=0)
    * 100
)

display(split_percentages.round(2))

split,train,val,test
class_name,,,
battery,70.02,15.04,14.94
biological,70.01,15.05,14.94
cardboard,69.97,15.01,15.01
clothes,70.00,15.00,15.00
glass,69.98,15.00,15.03
metal,70.00,15.00,15.00
paper,70.00,15.00,15.00
plastic,70.01,14.97,15.02
shoes,70.01,14.97,15.02


Overall split summary

In [33]:
split_summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "image_count": [
        len(train_df),
        len(val_df),
        len(test_df)
    ]
})

split_summary["percentage"] = (
    split_summary["image_count"]
    / len(df)
    * 100
)

display(split_summary)

,split,image_count,percentage
0,train,13833,69.997976
1,val,2964,14.998482
2,test,2965,15.003542


Sort manifests before saving

In [34]:
train_df = train_df.sort_values(
    ["class_name", "relative_path"]
).reset_index(drop=True)

val_df = val_df.sort_values(
    ["class_name", "relative_path"]
).reset_index(drop=True)

test_df = test_df.sort_values(
    ["class_name", "relative_path"]
).reset_index(drop=True)

Save manifests

In [35]:
train_path = MANIFEST_DIR / "train.csv"
val_path = MANIFEST_DIR / "val.csv"
test_path = MANIFEST_DIR / "test.csv"
summary_path = MANIFEST_DIR / "split_summary.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)
split_summary.to_csv(summary_path, index=False)

print("Saved:")
print(train_path)
print(val_path)
print(test_path)
print(summary_path)

Saved:
..\data\manifests\train.csv
..\data\manifests\val.csv
..\data\manifests\test.csv
..\data\manifests\split_summary.csv


Verify saved manifests

In [36]:
saved_train = pd.read_csv(train_path)
saved_val = pd.read_csv(val_path)
saved_test = pd.read_csv(test_path)

assert len(saved_train) == len(train_df)
assert len(saved_val) == len(val_df)
assert len(saved_test) == len(test_df)

assert set(saved_train["relative_path"]) == train_paths
assert set(saved_val["relative_path"]) == val_paths
assert set(saved_test["relative_path"]) == test_paths

print("Saved manifests verified successfully.")

Saved manifests verified successfully.


## Frozen Dataset Split

The dataset was split using stratified sampling with a fixed random seed
(`SEED = 42`) into:

- 70% training
- 15% validation
- 15% testing

The same manifests will be used for all deep-learning architectures to
ensure fair experimental comparison.

The validation set may be used for model selection, hyperparameter
tuning, early stopping, and checkpoint selection.

The test set is frozen and must not be used for preprocessing decisions,
hyperparameter tuning, architecture selection, or repeated model
development. It will be used only for final evaluation after model
configurations have been finalized.